# PR0503. Limpieza de datos sobre dataset de cultivos

In [7]:
from pyspark.sql import SparkSession

try: 
    spark = (SparkSession.builder.appName("PR0503")
              .master("spark://spark-master:7077")
              .getOrCreate()
            )

    print("SparkSession iniciada correctamente.")
except Exception as e:
    print("Error en la conexion")
    print(e)

sc = spark.sparkContext

SparkSession iniciada correctamente.


## Dataset 1: Datos para la predicción del rendimiento en cultivos

In [10]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import col, concat, concat_ws, substring, col, lit, upper, lpad

schema_crop = StructType([
    StructField("Crop", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Soil_Type", StringType(), True),
    StructField("Soil_pH", DoubleType(), True),
    StructField("Rainfall_mm", DoubleType(), True),
    StructField("Temperature_C", DoubleType(), True),
    StructField("Humidity_pct", DoubleType(), True),
    StructField("Fertilizer_Used_kg", DoubleType(), True),
    StructField("Irrigation", StringType(), True),
    StructField("Pesticides_Used_kg", DoubleType(), True),
    StructField("Planting_Density", DoubleType(), True),
    StructField("Previous_Crop", StringType(), True),
    StructField("Yield_ton_per_ha", DoubleType(), True)
])

df_crops = (spark.read
                .format("csv")
                .schema(schema_crop)
                .option("header", "true")
                .option("quote", "\"")
                .load("./crop_yield_dataset.csv")
           )

df_crops.printSchema()
df_crops.show(5)

root
 |-- Crop: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Soil_Type: string (nullable = true)
 |-- Soil_pH: double (nullable = true)
 |-- Rainfall_mm: double (nullable = true)
 |-- Temperature_C: double (nullable = true)
 |-- Humidity_pct: double (nullable = true)
 |-- Fertilizer_Used_kg: double (nullable = true)
 |-- Irrigation: string (nullable = true)
 |-- Pesticides_Used_kg: double (nullable = true)
 |-- Planting_Density: double (nullable = true)
 |-- Previous_Crop: string (nullable = true)
 |-- Yield_ton_per_ha: double (nullable = true)

+------+--------+---------+-------+-----------+-------------+------------+------------------+----------+------------------+----------------+-------------+----------------+
|  Crop|  Region|Soil_Type|Soil_pH|Rainfall_mm|Temperature_C|Humidity_pct|Fertilizer_Used_kg|Irrigation|Pesticides_Used_kg|Planting_Density|Previous_Crop|Yield_ton_per_ha|
+------+--------+---------+-------+-----------+-------------+------------+-------

### Creación de un ID único

In [11]:
df_eng = df_crops.withColumn("Crop_ID", 
    concat_ws("-", 
              concat(
                  lit("CODIGO_"),
                  lpad(substring(col("Region"), 8, 1), 3, "X")
              ),
              upper(col("Crop")) 
    )
)
df_eng.printSchema()
df_eng.show(5)

root
 |-- Crop: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Soil_Type: string (nullable = true)
 |-- Soil_pH: double (nullable = true)
 |-- Rainfall_mm: double (nullable = true)
 |-- Temperature_C: double (nullable = true)
 |-- Humidity_pct: double (nullable = true)
 |-- Fertilizer_Used_kg: double (nullable = true)
 |-- Irrigation: string (nullable = true)
 |-- Pesticides_Used_kg: double (nullable = true)
 |-- Planting_Density: double (nullable = true)
 |-- Previous_Crop: string (nullable = true)
 |-- Yield_ton_per_ha: double (nullable = true)
 |-- Crop_ID: string (nullable = false)

+------+--------+---------+-------+-----------+-------------+------------+------------------+----------+------------------+----------------+-------------+----------------+-----------------+
|  Crop|  Region|Soil_Type|Soil_pH|Rainfall_mm|Temperature_C|Humidity_pct|Fertilizer_Used_kg|Irrigation|Pesticides_Used_kg|Planting_Density|Previous_Crop|Yield_ton_per_ha|          Crop_ID|
+----

### Transformación matemática

In [13]:
from pyspark.sql.functions import log, round, bround

df_eng = df_eng.withColumn("Log_Rainfall", log(col("Rainfall_mm") + 1)) \
               .withColumn("Yield_Redondeado", round(col("Yield_ton_per_ha"), 1)) \
               .withColumn("Rendimiento_Bancario", bround(col("Yield_ton_per_ha"), 0)) \
               .drop("Rainfall_mm") \
               .drop("Yield_ton_per_ha")

df_eng.select(
    "Log_Rainfall",  
    "Yield_Redondeado", 
    "Rendimiento_Bancario"
).show(5)

+-----------------+----------------+--------------------+
|     Log_Rainfall|Yield_Redondeado|Rendimiento_Bancario|
+-----------------+----------------+--------------------+
|7.304112368059574|           101.5|               101.0|
|5.992464047441065|           127.4|               127.0|
|6.889489470175245|            69.0|                69.0|
|6.961580365677045|           169.1|               169.0|
|6.614189263371381|           118.7|               119.0|
+-----------------+----------------+--------------------+
only showing top 5 rows



### Comparación de insumos

In [15]:
from pyspark.sql.functions import greatest

df_eng = df_eng.withColumn("Max_Quimico_kg", greatest("Fertilizer_Used_kg", "Pesticides_Used_kg"))

df_eng.select(
    "Max_Quimico_kg",
    "Fertilizer_Used_kg",
    "Pesticides_Used_kg"
).show(5)

+--------------+------------------+------------------+
|Max_Quimico_kg|Fertilizer_Used_kg|Pesticides_Used_kg|
+--------------+------------------+------------------+
|         105.1|             105.1|              10.2|
|         221.8|             221.8|              35.5|
|          61.2|              61.2|              40.0|
|         257.8|             257.8|              42.7|
|         195.8|             195.8|              25.5|
+--------------+------------------+------------------+
only showing top 5 rows



### Simulación de fechas


In [18]:
from pyspark.sql.functions import to_date, date_add, month

df_eng = df_eng.withColumn("Fecha_Siembra", to_date(lit("2023-04-01"))) \
               .withColumn("Fecha_Estimada_Cosecha", date_add(col("Fecha_Siembra"), 150)) \
               .withColumn("Mes_cosecha", month(col("Fecha_Estimada_Cosecha")))


df_eng.select(
    "Fecha_Siembra",
    "Fecha_Estimada_Cosecha",
    "Mes_cosecha"
).show(5)

+-------------+----------------------+-----------+
|Fecha_Siembra|Fecha_Estimada_Cosecha|Mes_cosecha|
+-------------+----------------------+-----------+
|   2023-04-01|            2023-08-29|          8|
|   2023-04-01|            2023-08-29|          8|
|   2023-04-01|            2023-08-29|          8|
|   2023-04-01|            2023-08-29|          8|
|   2023-04-01|            2023-08-29|          8|
+-------------+----------------------+-----------+
only showing top 5 rows

